# RSNA Knee Abnormality Detection — Audit dữ liệu

Chạy tuần tự từ trên xuống. Kernel **CPU** là đủ, không cần GPU.

| Cell | Việc | Thời gian |
|---|---|---|
| 1–2 | Cấu hình, dò cấu trúc thư mục + CSV | vài giây |
| 3–4 | Quét header DICOM → parquet | 10–20 phút |
| 5 | Phân bố nhãn trên các study gold | vài giây |
| 6 | Phân tích báo cáo đa ngôn ngữ | vài giây |
| 7 | Phân tích ảnh | vài giây |
| 8 | Đối chiếu ảnh ↔ nhãn | vài giây |
| 9–10 | Chọn chuỗi + QC bằng mắt | 2–5 phút |

**Chạy lần đầu:** đặt `TEST_MODE = True` ở cell 1 để quét 200 series (~2 phút),
xác nhận không lỗi, rồi mới đổi thành `False`.


## 1 — Cấu hình và import

In [ ]:
import os, re, sys, warnings, unicodedata, multiprocessing as mp
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 200)
%matplotlib inline

# ===================== CHỈNH Ở ĐÂY =====================
TEST_MODE = True          # True = chỉ quét 200 series để thử. Đổi False khi chạy thật.
N_QC_STUDIES = 12         # số study xuất montage ở cell 10
SAMPLES_PER_SERIES = 3    # số file đọc header mỗi series (đầu/giữa/cuối)
# =======================================================

OUT = Path("/kaggle/working")
OUT.mkdir(parents=True, exist_ok=True)

# Tự dò thư mục competition — tên có thể khác nếu bạn mount dataset riêng
cands = [p for p in Path("/kaggle/input").glob("*") if p.is_dir()]
ROOT = next((p for p in cands if "knee" in p.name.lower()), None)
if ROOT is None:
    ROOT = cands[0] if cands else None

print("Các dataset đang mount:")
for p in cands:
    print(f"   {p}")
print(f"\n>> ROOT = {ROOT}")
print(f">> TEST_MODE = {TEST_MODE}")
assert ROOT is not None, "Không thấy dataset nào. Add competition data vào notebook trước."

## 2 — Cấu trúc thư mục và các file CSV

Đây là bước **không được bỏ qua**: tên 12 cột nhãn phải lấy từ `sample_submission.csv`
của chính cuộc thi, không lấy từ mô tả của bên thứ ba.

In [ ]:
print("="*74); print("CÂY THƯ MỤC"); print("="*74)
for p in sorted(ROOT.iterdir()):
    print(f"  {p.name}{'/' if p.is_dir() else ''}")
    if p.is_dir():
        kids = sorted(p.iterdir())
        for k in kids[:3]:
            print(f"      {k.name}{'/' if k.is_dir() else ''}")
            if k.is_dir():
                gk = sorted(k.iterdir())[:2]
                for g in gk:
                    print(f"          {g.name}{'/' if g.is_dir() else ''}")
        if len(kids) > 3:
            print(f"      ... (tổng {len(kids)})")

CSV = {}
print("\n" + "="*74); print("CÁC FILE CSV"); print("="*74)
for c in sorted(ROOT.rglob("*.csv")):
    try:
        df = pd.read_csv(c)
    except Exception as e:
        print(f"\n  {c.name}: KHÔNG ĐỌC ĐƯỢC — {e}"); continue
    CSV[c.name] = df
    print(f"\n  {c.relative_to(ROOT)}   shape={df.shape}")
    print(f"    cột ({len(df.columns)}): {list(df.columns)}")
    print(f"    NaN: {df.isna().sum().to_dict()}")
    display(df.head(3))

# Xác định 12 cột nhãn từ sample_submission
SUB_KEY = next((k for k in CSV if "sample" in k.lower() and "sub" in k.lower()), None)
LABEL_COLS, ID_COLS = [], []
if SUB_KEY:
    ss = CSV[SUB_KEY]
    ID_COLS = [c for c in ss.columns if re.search(r"id$|uid$|^study", c, re.I)]
    LABEL_COLS = [c for c in ss.columns if c not in ID_COLS]
    print(f"\n>> Cột định danh: {ID_COLS}")
    print(f">> {len(LABEL_COLS)} CỘT NHÃN: {LABEL_COLS}")
    if len(LABEL_COLS) != 12:
        print(f"   !! Kỳ vọng 12, thấy {len(LABEL_COLS)} — kiểm tra lại thủ công.")
else:
    print("\n   !! Không thấy sample_submission.csv")

TRAIN_KEY = next((k for k in CSV if re.match(r"train", k, re.I)
                  and any(c in CSV[k].columns for c in LABEL_COLS)), None) \
            or next((k for k in CSV if re.match(r"train", k, re.I)), None)
print(f">> File train: {TRAIN_KEY}")

## 3 — Hàm suy luận từ DICOM header

Nguyên tắc: ưu tiên **trường số do máy ghi** (`ImageOrientationPatient`, `TR`/`TE`,
`ScanOptions`) hơn **text do người đặt** (`SeriesDescription`). 16 site là 16 quy ước
đặt tên ở 12 thứ tiếng, còn hình học thì chỉ có một.

In [ ]:
FATSAT_RE = re.compile("|".join([
    r"\bFS\b", r"FATSAT", r"FAT[ _-]?SAT", r"\bSAT\b", r"SPAIR", r"SPIR",
    r"STIR", r"TIRM", r"IDEAL", r"MDIXON", r"M[ _-]?DIXON", r"DIXON",
    r"\bW\b", r"WATER", r"CHESS", r"FATSUP", r"FSAT",
]), re.IGNORECASE)

EXCLUDE_RE = re.compile(
    r"LOCALIZER|SCOUT|SURVEY|CALIBRATION|REPORT|SCREEN[ _-]?SAVE|"
    r"DERIVED|SUBTRACT|MIP\b|3PLANE", re.IGNORECASE)

LATERALITY_RE = {
 "L": re.compile("|".join([r"\bLEFT\b", r"\bLT\b", r"\bLINKS\b", r"\bGAUCHE\b",
      r"\bIZQ", r"\bSINISTR", r"\bESQUERD", r"\bSOL\b", r"\bVENSTRE\b",
      r"\bLEWE", r"\bLEVÝ", r"\bLEVA", r"\bSX\b"]), re.I),
 "R": re.compile("|".join([r"\bRIGHT\b", r"\bRT\b", r"\bRECHTS\b", r"\bDROIT",
      r"\bDER(ECH)?", r"\bDESTR", r"\bDIREIT", r"\bSAĞ\b", r"\bHØYRE\b",
      r"\bPRAW", r"\bPRAVÝ", r"\bDESNA", r"\bDX\b"]), re.I),
}

TAGS = ["StudyInstanceUID","SeriesInstanceUID","SeriesNumber","InstanceNumber",
    "SeriesDescription","StudyDescription","ProtocolName","SequenceName","Modality",
    "BodyPartExamined","ImageLaterality","Laterality","PatientSex","PatientAge",
    "Manufacturer","ManufacturerModelName","InstitutionName","StationName",
    "MagneticFieldStrength","ScanningSequence","SequenceVariant","ScanOptions",
    "MRAcquisitionType","RepetitionTime","EchoTime","InversionTime","FlipAngle",
    "EchoTrainLength","PixelBandwidth","Rows","Columns","PixelSpacing",
    "SliceThickness","SpacingBetweenSlices","ImageOrientationPatient",
    "ImagePositionPatient","PhotometricInterpretation","PixelRepresentation",
    "BitsStored","RescaleSlope","RescaleIntercept","NumberOfFrames"]


def compute_plane(iop):
    """Mặt phẳng cắt, suy từ hình học. KHÔNG dùng SeriesDescription cho việc này."""
    if iop is None: return "UNKNOWN"
    try:
        v = [float(x) for x in iop]
        if len(v) != 6: return "UNKNOWN"
    except (TypeError, ValueError): return "UNKNOWN"
    n = np.cross(np.array(v[:3]), np.array(v[3:]))
    return {0: "SAG", 1: "COR", 2: "AX"}[int(np.argmax(np.abs(n)))]


def compute_obliquity_deg(iop):
    """Góc lệch so với mặt phẳng chuẩn. Cao = chuỗi cắt xiên."""
    if iop is None: return np.nan
    try:
        v = [float(x) for x in iop]
        if len(v) != 6: return np.nan
    except (TypeError, ValueError): return np.nan
    n = np.cross(np.array(v[:3]), np.array(v[3:]))
    n = n / (np.linalg.norm(n) + 1e-9)
    return float(np.degrees(np.arccos(np.clip(np.max(np.abs(n)), 0, 1))))


def infer_weighting(tr, te, ti, seq):
    """Loại trọng số. Ranh giới là quy ước lâm sàng — kiểm tra lại phân bố
    TR/TE thực tế ở cell 7 trước khi tin tuyệt đối."""
    s = str(seq or "").upper()
    if ti is not None and not pd.isna(ti) and 0 < float(ti) < 400 and "IR" in s:
        return "STIR"
    if tr is None or te is None or pd.isna(tr) or pd.isna(te): return "UNKNOWN"
    tr, te = float(tr), float(te)
    if te >= 60: return "T2"
    if tr >= 1000 and te < 45: return "PD"
    if tr < 800 and te < 30: return "T1"
    return "UNKNOWN"


def infer_fatsat(opts, seq, ti, desc, seqname):
    o = str(opts or "").upper()
    if any(k in o for k in ("FS","SFS","FAT","SPAIR","SPIR","CHEMSAT")): return True
    if "IR" in str(seq or "").upper() and ti is not None and not pd.isna(ti) \
       and 0 < float(ti) < 400: return True
    return any(t and FATSAT_RE.search(str(t)) for t in (desc, seqname))


def infer_laterality(img_lat, lat, body, desc, sdesc):
    """Bắt buộc phải biết: có nhãn phân biệt medial/lateral, không chuẩn hóa
    thì model học nhầm bên."""
    for v in (img_lat, lat):
        s = str(v or "").strip().upper()
        if s in ("L", "R"): return s
    for t in (body, desc, sdesc):
        if not t: continue
        l, r = bool(LATERALITY_RE["L"].search(str(t))), bool(LATERALITY_RE["R"].search(str(t)))
        if l != r: return "L" if l else "R"     # chỉ nhận khi không mơ hồ
    return "UNKNOWN"


def _get(ds, tag, d=None):
    v = getattr(ds, tag, d)
    if v is None: return d
    if isinstance(v, pydicom.multival.MultiValue): return list(v)
    if isinstance(v, pydicom.valuerep.PersonName): return str(v)
    return v

# Kiểm tra nhanh
assert compute_plane([0,1,0,0,0,-1]) == "SAG"
assert compute_plane([1,0,0,0,0,-1]) == "COR"
assert compute_plane([1,0,0,0,1,0])  == "AX"
assert infer_weighting(3000,80,None,"SE") == "T2"
assert infer_weighting(3000,30,None,"SE") == "PD"
assert infer_weighting(600,15,None,"SE")  == "T1"
assert infer_weighting(4000,40,150,"IR")  == "STIR"
assert infer_fatsat("FS","SE",None,"","") is True
assert infer_fatsat("","SE",None,"PD TIRM SAG","") is True
assert infer_fatsat("","SE",None,"T1 SAG","") is False
assert infer_laterality(None,None,"KNIE","MRT Knie rechts","") == "R"
assert infer_laterality(None,None,None,"Sol diz MR","") == "L"
print("Các hàm suy luận: OK")

In [ ]:
# ---- SỬA ROOT ----
ROOT = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
print("ROOT =", ROOT)
print([p.name for p in sorted(ROOT.iterdir())])

# ---- DÙNG METADATA CÓ SẴN THAY VÌ HEURISTIC ----
TS = CSV["train_series.csv"].copy()
TS["plane"] = TS["Anatomical_Plane"].map(
    {"Sagittal": "SAG", "Coronal": "COR", "Axial": "AX"})
TS["fluid"] = TS["Fluid_Sensitive"].astype(int)
TS["fat"]   = TS["Fat_Suppression"].astype(int)
TS["priority"] = TS["fluid"] * 2 + TS["fat"]   # 3 = fluid+fatsat (tốt nhất)

print(f"\nSeries: {len(TS):,} | Study: {TS.StudyInstanceUID.nunique():,}")
print(f"Series/study: {TS.groupby('StudyInstanceUID').size().describe().to_string()}")

print("\nBẢNG CHÉO: mặt phẳng × (fluid, fat-sat)")
display(pd.crosstab(TS["plane"],
                    [TS["fluid"], TS["fat"]], margins=True))

# Độ phủ chuỗi ưu tiên ở cấp study
best = TS[TS["priority"] == 3]
cov = pd.DataFrame(index=sorted(TS.StudyInstanceUID.unique()))
for pl in ["SAG", "COR", "AX"]:
    cov[pl] = cov.index.isin(best[best.plane == pl].StudyInstanceUID)
cov["all3"] = cov[["SAG", "COR", "AX"]].all(axis=1)

print("\nĐỘ PHỦ chuỗi fluid-sensitive fat-suppressed:")
for c in ["SAG", "COR", "AX", "all3"]:
    n = int(cov[c].sum())
    print(f"  {c:5s}: {n:5d}/{len(cov)}  ({n/len(cov):6.1%})")
print(f"\n  !! {len(cov)-int(cov['all3'].sum())} study thiếu -> cần fallback")

# Study nào có nhãn gold?
TR = CSV["train.csv"]
LBL = ['ACL','MCL','Medial Meniscus','Lateral Meniscus','Medial OA',
       'Lateral OA','PF OA','Effusion','Synovitis',"Baker's",
       'Contusion','Fracture']
G = TR.dropna(subset=LBL, how="all")
print(f"\nStudy gold: {len(G)} / {len(TR)}  ({len(G)/len(TR):.2%})")

st = pd.DataFrame([{
    "label": c, "pos": int((G[c]==1).sum()), "neg": int((G[c]==0).sum()),
    "prevalence": (G[c]==1).mean(),
    "auc_unstable": (G[c]==1).sum() < 5 or (G[c]==0).sum() < 5
} for c in LBL]).sort_values("pos")
display(st)

# Gold có đại diện không?
cov_g = cov.loc[cov.index.isin(G.StudyInstanceUID)]
print(f"\nĐộ phủ all3 — gold: {cov_g['all3'].mean():.1%} | "
      f"toàn bộ: {cov['all3'].mean():.1%}")
print(">> Chênh nhiều nghĩa là 58 ca gold không đại diện -> validation lệch.")

## 4 — Quét header DICOM

Lấy mẫu 3 file mỗi series (đầu / giữa / cuối) thay vì đọc hết ~819k file — đủ để bắt
sự không đồng nhất trong series mà nhanh hơn nhiều lần.

Đặt `TEST_MODE = False` ở cell 1 khi chạy thật.

In [ ]:
def scan_series(series_dir):
    p = Path(series_dir)
    files = sorted(f for f in p.iterdir() if f.is_file())
    if not files:
        return {"series_dir": series_dir, "error": "empty_dir"}
    n = len(files)
    idxs = sorted({0, n // 2, n - 1})[:SAMPLES_PER_SERIES]
    samples, errs = [], []
    for i in idxs:
        try:
            samples.append(pydicom.dcmread(str(files[i]), stop_before_pixels=True, force=True))
        except Exception as e:
            errs.append(f"{files[i].name}: {type(e).__name__}")
    if not samples:
        return {"series_dir": series_dir, "n_files": n, "error": "; ".join(errs) or "unreadable"}

    ds = samples[0]
    row = {"series_dir": series_dir, "n_files": n, "error": None}
    for t in TAGS: row[t] = _get(ds, t)

    ps = row.pop("PixelSpacing", None)
    row["pixel_spacing_row"] = float(ps[0]) if isinstance(ps, list) and ps else np.nan
    row["pixel_spacing_col"] = float(ps[1]) if isinstance(ps, list) and len(ps) > 1 else np.nan

    nf = row.get("NumberOfFrames")
    row["is_multiframe"] = bool(nf) and int(nf) > 1
    row["n_slices"] = int(nf) * n if row["is_multiframe"] else n

    iop = row.pop("ImageOrientationPatient", None)
    row["plane"] = compute_plane(iop)
    row["obliquity_deg"] = compute_obliquity_deg(iop)

    row["weighting"] = infer_weighting(row.get("RepetitionTime"), row.get("EchoTime"),
                                       row.get("InversionTime"), row.get("ScanningSequence"))
    row["fatsat"] = infer_fatsat(row.get("ScanOptions"), row.get("ScanningSequence"),
                                 row.get("InversionTime"), row.get("SeriesDescription"),
                                 row.get("SequenceName"))
    row["laterality"] = infer_laterality(row.get("ImageLaterality"), row.get("Laterality"),
                                         row.get("BodyPartExamined"),
                                         row.get("SeriesDescription"), row.get("StudyDescription"))
    row["excluded_by_desc"] = bool(EXCLUDE_RE.search(str(row.get("SeriesDescription") or "")))

    # Cờ tiền xử lý — mỗi cái bỏ qua là một lỗi âm thầm
    row["monochrome1"] = str(row.get("PhotometricInterpretation") or "") == "MONOCHROME1"
    row["signed_pixels"] = int(row.get("PixelRepresentation") or 0) == 1
    row["needs_rescale"] = not ((row.get("RescaleSlope") in (None,1,1.0,"1")) and
                                (row.get("RescaleIntercept") in (None,0,0.0,"0")))

    pos = []
    for s in samples:
        ipp = _get(s, "ImagePositionPatient")
        if isinstance(ipp, list) and len(ipp) == 3:
            pos.append(np.array([float(x) for x in ipp]))
    row["z_span_mm"] = float(np.linalg.norm(pos[-1] - pos[0])) if len(pos) >= 2 else np.nan
    row["inconsistent_dims"] = len({(_get(s,"Rows"), _get(s,"Columns")) for s in samples}) > 1
    row.pop("ImagePositionPatient", None)

    if not row.get("StudyInstanceUID"):  row["StudyInstanceUID"]  = p.parent.name
    if not row.get("SeriesInstanceUID"): row["SeriesInstanceUID"] = p.name
    return row


def discover_series_dirs(root):
    """Dò thư mục chứa DICOM. Không giả định trước cấu trúc study/series/*.dcm."""
    dirs, depths = [], Counter()
    for dp, _, fns in os.walk(root):
        if not fns: continue
        ok = any(f.lower().endswith(".dcm") for f in fns)
        if not ok and len(fns) > 3:
            try:
                pydicom.dcmread(str(Path(dp)/fns[0]), stop_before_pixels=True,
                                force=True, specific_tags=["SOPClassUID"])
                ok = True
            except Exception:
                ok = False
        if ok:
            dirs.append(dp)
            depths[len(Path(dp).relative_to(root).parts)] += 1
    return sorted(dirs), (depths.most_common(1)[0][0] if depths else 0)


series_dirs, depth = discover_series_dirs(ROOT)
print(f"Tìm thấy {len(series_dirs):,} thư mục series (độ sâu phổ biến: {depth})")
assert series_dirs, "Không thấy DICOM nào."
if TEST_MODE:
    series_dirs = series_dirs[:200]
    print(f"TEST_MODE — chỉ quét {len(series_dirs)} series")

n_workers = os.cpu_count() or 4
print(f"\nQuét bằng {n_workers} tiến trình...")
rows, done = [], 0
ctx = mp.get_context("fork")          # fork để tiến trình con kế thừa hàm đã định nghĩa
with ctx.Pool(n_workers) as pool:
    for r in pool.imap_unordered(scan_series, series_dirs, chunksize=8):
        rows.append(r); done += 1
        if done % 500 == 0 or done == len(series_dirs):
            print(f"  {done:,}/{len(series_dirs):,}", flush=True)

S = pd.DataFrame(rows)
errors = S[S["error"].notna()]
S = S[S["error"].isna()].drop(columns=["error"])
if len(errors):
    print(f"\n  !! {len(errors)} series lỗi")
    display(errors.head())

for c in S.columns:
    if S[c].dtype == object:
        S[c] = S[c].astype(str).replace({"None": None, "nan": None})
S.to_parquet(OUT/"series_meta.parquet", index=False)

ST = (S.groupby("StudyInstanceUID").agg(
        n_series=("SeriesInstanceUID","nunique"), n_slices_total=("n_slices","sum"),
        planes=("plane", lambda s: "|".join(sorted(set(s)))),
        n_planes=("plane", lambda s: len(set(s)-{"UNKNOWN"})),
        n_fatsat=("fatsat","sum"),
        lateralities=("laterality", lambda s: "|".join(sorted(set(s)))),
        manufacturer=("Manufacturer","first"), institution=("InstitutionName","first"),
        model=("ManufacturerModelName","first"),
        field_strength=("MagneticFieldStrength","first")).reset_index())
fs = S[S["fatsat"].astype(bool)]
for pl in ("SAG","COR","AX"):
    ok = set(fs[fs["plane"]==pl]["StudyInstanceUID"])
    ST[f"fatsat_{pl.lower()}"] = ST["StudyInstanceUID"].isin(ok)
ST["fatsat_all3"] = ST["fatsat_sag"] & ST["fatsat_cor"] & ST["fatsat_ax"]
ST.to_parquet(OUT/"study_meta.parquet", index=False)

print(f"\nĐã lưu: {len(S):,} series / {len(ST):,} study")
display(S.head(3))

## 5 — Phân bố nhãn trên các study gold

**Bảng quan trọng nhất của toàn bộ audit.** Nhãn nào có dưới 5 mẫu ở một lớp thì AUC
trên đó là nhiễu thuần túy — nhưng macro-AUC vẫn tính nó ngang các nhãn khác.

In [ ]:
assert TRAIN_KEY, "Không xác định được file train."
TR = CSV[TRAIN_KEY]
present = [c for c in LABEL_COLS if c in TR.columns]

if not present:
    print(f"{TRAIN_KEY} không chứa cột nhãn. Cột hiện có: {list(TR.columns)}")
    print(">> Nhãn có thể nằm ở file CSV khác — xem lại output cell 2.")
else:
    G = TR.dropna(subset=present, how="all")
    print(f"Tổng dòng          : {len(TR):,}")
    print(f"Dòng CÓ nhãn       : {len(G):,}")
    print(f"Dòng KHÔNG nhãn    : {len(TR)-len(G):,}")
    print(f"Tỉ lệ có nhãn      : {len(G)/max(len(TR),1):.2%}")
    print("\n>> Tỉ lệ này quyết định toàn bộ thiết kế: đây là bài weak supervision,")
    print("   không phải bài classification thông thường.\n")

    st = []
    for c in present:
        s = G[c].dropna()
        pos, neg = int((s==1).sum()), int((s==0).sum())
        st.append({"label": c, "n_gold": pos+neg, "pos": pos, "neg": neg,
                   "prevalence": pos/(pos+neg) if pos+neg else np.nan,
                   "auc_unstable": pos < 5 or neg < 5})
    ST_LBL = pd.DataFrame(st).sort_values("pos")
    display(ST_LBL.style.format({"prevalence":"{:.3f}"})
            .map(lambda v: "background-color:#ffd6d6" if v is True else "",
                 subset=["auc_unstable"]))

    risky = ST_LBL[ST_LBL["auc_unstable"]]
    if len(risky):
        print(f"!! {len(risky)} nhãn dưới 5 mẫu ở một lớp: {list(risky['label'])}")
        print("   AUC trên chúng gần như vô nghĩa — đừng để chúng lái quyết định model.")

    print("\nTương quan giữa các nhãn (gợi ý dùng head chung / multi-task):")
    corr = G[present].astype(float).corr()
    display(corr.round(2).style.background_gradient(cmap="RdBu_r", vmin=-1, vmax=1))
    for i, a in enumerate(present):
        for b in present[i+1:]:
            v = corr.loc[a, b]
            if pd.notna(v) and abs(v) > 0.5:
                print(f"   {a} <-> {b}: {v:+.2f}")

    print("\nSố nhãn dương mỗi study:")
    print(G[present].astype(float).sum(axis=1).value_counts().sort_index().to_string())

    fig, ax = plt.subplots(figsize=(9, 4))
    d = ST_LBL.sort_values("pos")
    ax.barh(d["label"], d["pos"], color="#c44", label="dương")
    ax.barh(d["label"], d["neg"], left=d["pos"], color="#bbb", label="âm")
    ax.axvline(5, ls="--", c="k", lw=1)
    ax.set_xlabel("số study gold"); ax.legend()
    ax.set_title("Phân bố nhãn trên tập gold (vạch đứt = ngưỡng 5 mẫu)")
    plt.tight_layout(); plt.show()

## 6 — Báo cáo chẩn đoán

Phần này quyết định thiết kế report labeler. Chú ý hai bảng cuối: **từ phủ định**
và **từ chỉ mức độ**. Ground truth dùng ngưỡng độ nặng và xếp ca lưỡng lự thành
âm tính, nên labeler phải bắt được mức độ chứ không chỉ bắt sự hiện diện.

In [ ]:
SCRIPTS = [("CYRILLIC",0x0400,0x04FF),("GREEK",0x0370,0x03FF),("ARABIC",0x0600,0x06FF),
           ("HEBREW",0x0590,0x05FF),("HAN",0x4E00,0x9FFF),("HIRAGANA",0x3040,0x309F),
           ("KATAKANA",0x30A0,0x30FF),("HANGUL",0xAC00,0xD7AF),("THAI",0x0E00,0x0E7F),
           ("DEVANAGARI",0x0900,0x097F)]
STOPWORDS = {
 "en":["the","and","of","with","is","no","there","normal"],
 "es":["de","el","la","con","sin","se","en","los"],
 "pt":["de","do","da","com","sem","não","para","dos"],
 "fr":["de","le","la","des","avec","sans","est","une"],
 "de":["der","die","das","und","mit","ohne","ist","kein"],
 "it":["di","il","la","con","senza","del","non","nella"],
 "nl":["de","het","een","met","zonder","geen","van","is"],
 "tr":["ve","ile","bir","olarak","yok","var","sağ","sol"],
 "pl":["i","w","na","z","nie","jest","oraz","bez"],
}

def guess_language(text):
    cnt = Counter()
    for ch in text[:600]:
        cp = ord(ch)
        for nm, lo, hi in SCRIPTS:
            if lo <= cp <= hi: cnt[nm] += 1; break
        else:
            if ch.isalpha(): cnt["LATIN"] += 1
    if not cnt: return "UNKNOWN"
    script = cnt.most_common(1)[0][0]
    if script != "LATIN": return script
    wc = Counter(re.findall(r"\b\w+\b", text.lower())[:400])
    sc = {l: sum(wc.get(w,0) for w in sw) for l, sw in STOPWORDS.items()}
    best, s = max(sc.items(), key=lambda kv: kv[1])
    return best if s >= 3 else "LATIN?"

TEXT_COL = RPT_KEY = None
for k, v in CSV.items():
    for c in v.columns:
        if re.search(r"report|text|finding|impression|narrative", c, re.I):
            TEXT_COL, RPT_KEY = c, k; break
    if TEXT_COL: break

if TEXT_COL is None:
    print("Không thấy cột báo cáo — kiểm tra tay danh sách cột ở cell 2.")
else:
    print(f"Nguồn: {RPT_KEY}, cột '{TEXT_COL}'")
    txt = CSV[RPT_KEY][TEXT_COL].dropna().astype(str)
    print(f"Báo cáo không rỗng: {len(txt):,} / {len(CSV[RPT_KEY]):,}\n")

    L = txt.str.len()
    print("Độ dài (ký tự):"); print(L.describe(percentiles=[.01,.05,.5,.95,.99]).to_string())
    if (L < 50).sum():
        print(f"\n!! {(L<50).sum()} báo cáo dưới 50 ký tự — có thể bị cắt cụt:")
        for t in txt[L < 50].head(5): print(f"   {t!r}")

    smp = txt.sample(min(len(txt), 3000), random_state=0)
    langs = smp.map(guess_language); lc = langs.value_counts()
    print(f"\nNgôn ngữ (heuristic, {len(lc)} nhóm — BTC công bố 12):")
    display(pd.DataFrame({"n": lc, "pct": (lc/len(smp)*100).round(1)}))

    print("\n" + "="*74)
    print("MẪU BÁO CÁO — ĐỌC BẰNG MẮT, ĐỪNG BỎ QUA BƯỚC NÀY")
    print("="*74)
    for lg in lc.index[:8]:
        t = smp[langs == lg].iloc[0]
        print(f"\n[{lg}]\n{t[:500]}{'...' if len(t)>500 else ''}")

    print("\n\nCó heading cấu trúc không?")
    for pat, nm in [(r"FINDINGS|HALLAZGOS|BEFUND|CONSTATATION","findings"),
                    (r"IMPRESSION|CONCLUS|IMPRESI|BEURTEIL","impression"),
                    (r"TECHNIQUE|TÉCNICA|TECHNIK","technique")]:
        print(f"  {nm:12s}: {txt.str.contains(pat, case=False, regex=True).mean():6.1%}")

    print("\nTừ PHỦ ĐỊNH (labeler phải có 3 trạng thái: dương / âm / không nhắc):")
    for w in ["no","not","without","absence","negative","unremarkable","intact",
              "sin","kein","sans","senza","geen","yok"]:
        p = txt.str.contains(rf"\b{re.escape(w)}\b", case=False).mean()
        if p > 0.01: print(f"  {w!r:14s}: {p:6.1%}")

    print("\nTừ chỉ MỨC ĐỘ (ground truth có ngưỡng độ nặng!):")
    for w in ["mild","minimal","trace","small","moderate","severe","large",
              "leve","leicht","mäßig","légère","grave"]:
        p = txt.str.contains(rf"\b{re.escape(w)}", case=False).mean()
        if p > 0.005: print(f"  {w!r:12s}: {p:6.1%}")

## 7 — Dữ liệu ảnh

In [ ]:
S  = pd.read_parquet(OUT/"series_meta.parquet")
ST = pd.read_parquet(OUT/"study_meta.parquet")
for c in ["fatsat","monochrome1","signed_pixels","needs_rescale","is_multiframe",
          "inconsistent_dims","excluded_by_desc"]:
    if c in S.columns:
        S[c] = S[c].astype(str).str.lower().isin(["true","1"])

print(f"Study: {len(ST):,}   Series: {len(S):,}   Slice: {int(S['n_slices'].sum()):,}\n")

print("Slice mỗi series:")
print(S["n_slices"].describe(percentiles=[.01,.05,.5,.95,.99]).to_string())
print(f"\nTỉ lệ max/min = {S['n_slices'].max()/max(S['n_slices'].min(),1):.0f}x")
print(">> Chênh lớn thế này thì phải sampling số lượng CỐ ĐỊNH, không load hết.\n")

print("BẢNG CHÉO: mặt phẳng × fat-sat")
display(pd.crosstab(S["plane"], S["fatsat"], margins=True))
print("BẢNG CHÉO: mặt phẳng × trọng số")
display(pd.crosstab(S["plane"], S["weighting"], margins=True))

print("Độ phủ chuỗi ưu tiên (fluid-sensitive fat-sat), cấp STUDY:")
for c, nm in [("fatsat_sag","SAG fat-sat"),("fatsat_cor","COR fat-sat"),
              ("fatsat_ax","AX fat-sat"),("fatsat_all3","ĐỦ CẢ 3")]:
    if c in ST.columns:
        n = int(ST[c].sum()); print(f"  {nm:14s}: {n:5d}/{len(ST)}  ({n/len(ST):6.1%})")
if "fatsat_all3" in ST.columns:
    miss = len(ST) - int(ST["fatsat_all3"].sum())
    print(f"\n!! {miss} study THIẾU chuỗi ưu tiên -> bắt buộc có fallback.")
    print("   Đừng bỏ chúng: vẫn là dữ liệu train hợp lệ.")

print("\nLATERALITY:")
lat = S["laterality"].value_counts(); print(lat.to_string())
unk = int(lat.get("UNKNOWN", 0))
if unk/len(S) > 0.05:
    print(f"\n!! {unk/len(S):.1%} không xác định được trái/phải.")
    print("   Không chuẩn hóa thì model học nhầm bên medial/lateral.")
    print("   Phương án: (a) mở rộng từ điển đa ngôn ngữ, hoặc")
    print("              (b) train một classifier trái/phải nhỏ từ chính ảnh.")
conf = ST[ST["lateralities"].str.contains(r"L\|R", na=False)]
if len(conf): print(f"!! {len(conf)} study có series mâu thuẫn trái/phải.")

print("\nKích thước voxel — dùng để chọn mm/pixel mục tiêu khi resample:")
display(S[["pixel_spacing_row","pixel_spacing_col","SliceThickness"]]
        .astype(float).describe(percentiles=[.05,.5,.95]))
print(">> Resample theo mm/pixel, KHÔNG resize theo số pixel.")

print("\nCỜ TIỀN XỬ LÝ — mỗi cái bỏ qua là một lỗi âm thầm:")
for c, note in [("monochrome1","thang xám đảo -> phải invert"),
                ("signed_pixels","pixel có dấu"),
                ("needs_rescale","phải áp RescaleSlope/Intercept"),
                ("is_multiframe","1 file nhiều frame"),
                ("inconsistent_dims","kích thước đổi giữa series"),
                ("excluded_by_desc","localizer/scout -> loại")]:
    if c in S.columns:
        n = int(S[c].sum()); print(f"  {c:20s}: {n:5d} ({n/len(S):5.1%})  — {note}")

print("\nSITE / MÁY — nguồn domain shift, test đến từ site khác:")
for c in ["InstitutionName","Manufacturer","ManufacturerModelName","MagneticFieldStrength"]:
    if c in S.columns and S[c].notna().any():
        vc = S[c].value_counts()
        print(f"\n  {c} ({vc.nunique()} giá trị):"); print(vc.head(15).to_string())
    else:
        print(f"\n  {c}: TRỐNG (đã ẩn danh)")
print("\n>> Nếu InstitutionName trống, dùng Manufacturer+Model+FieldStrength")
print("   làm proxy cho site khi chia GroupKFold.")

fig, ax = plt.subplots(1, 3, figsize=(15, 3.4))
S["n_slices"].plot.hist(bins=60, ax=ax[0]); ax[0].set_title("slice / series")
S["pixel_spacing_row"].astype(float).plot.hist(bins=60, ax=ax[1]); ax[1].set_title("pixel spacing (mm)")
S["SliceThickness"].astype(float).plot.hist(bins=60, ax=ax[2]); ax[2].set_title("slice thickness (mm)")
plt.tight_layout(); plt.show()

print("\nSeriesDescription phổ biến — kiểm tra logic chọn chuỗi:")
print(S["SeriesDescription"].value_counts().head(40).to_string())

## 8 — Đối chiếu ảnh ↔ nhãn

Câu hỏi cần trả lời: **các study gold có đại diện cho phần còn lại không?**
Nếu chúng khác nhiều về mặt kỹ thuật thì mọi validation của bạn sẽ lệch.

In [ ]:
stp = OUT/"study_meta.parquet"
if TRAIN_KEY and stp.exists():
    ST = pd.read_parquet(stp); TR = CSV[TRAIN_KEY]
    idc = next((c for c in TR.columns if re.search(r"study.*(uid|id)|^id$", c, re.I)), None)
    print(f"Cột ID trong {TRAIN_KEY}: {idc}")

    img = set(ST["StudyInstanceUID"].astype(str))
    csvi = set(TR[idc].astype(str)) if idc else set()
    print(f"  Study có ảnh         : {len(img):,}")
    print(f"  Study trong CSV      : {len(csvi):,}")
    print(f"  Giao nhau            : {len(img & csvi):,}")
    print(f"  Có ảnh, thiếu CSV    : {len(img - csvi):,}")
    print(f"  Có CSV, thiếu ảnh    : {len(csvi - img):,}")

    join_col = "StudyInstanceUID"
    if not (img & csvi):
        print("\n!! KHÔNG GIAO NHAU — thử khớp bằng token cuối của UID:")
        ST["_short"] = ST["StudyInstanceUID"].astype(str).str.split(".").str[-1]
        if set(ST["_short"]) & csvi:
            join_col = "_short"; print("   Khớp được. Dùng token cuối.")
        else:
            print("   Vẫn không khớp — kiểm tra tay.")

    if idc and (set(ST[join_col].astype(str)) & csvi):
        M = ST.merge(TR, left_on=join_col, right_on=idc, how="left")
        pres = [c for c in LABEL_COLS if c in M.columns]
        if pres:
            M["has_gold"] = M[pres].notna().any(axis=1)
            print("\nStudy gold có khác study thường về mặt kỹ thuật không?")
            cols = [c for c in ["n_series","n_slices_total","fatsat_all3","n_planes"] if c in M.columns]
            display(M.groupby("has_gold")[cols].mean().round(2))
            print(">> Nếu khác nhiều, 58 ca gold KHÔNG đại diện -> validation lệch.")
            for c in ["manufacturer","institution","model"]:
                if c in M.columns and M[c].notna().any():
                    print(f"\nPhân bố gold theo {c}:")
                    display(pd.crosstab(M[c], M["has_gold"], normalize="columns").round(3))
                    break

## 9 — Chọn chuỗi ưu tiên

Điểm số phản ánh nguyên tắc: bệnh lý = nước, nên chuỗi nhạy dịch + triệt mỡ là tốt nhất.
Nhưng **luôn có fallback** — ~400 study thiếu chuỗi ưu tiên và chúng vẫn là dữ liệu
train hợp lệ.

In [ ]:
PLANES = ["SAG","COR","AX"]

def score_series(r):
    if r.get("excluded_by_desc"): return (-1, 0, "loại: localizer/scout")
    fat = bool(r.get("fatsat")); w = str(r.get("weighting") or "UNKNOWN")
    fluid = w in ("PD","T2","STIR")
    if fluid and fat:  sc, why = 100, f"fluid-sensitive fat-sat ({w})"
    elif w == "STIR":  sc, why = 90,  "STIR"
    elif fluid:        sc, why = 60,  f"fluid-sensitive, KHÔNG fat-sat ({w})"
    elif fat:          sc, why = 40,  f"fat-sat, trọng số không rõ ({w})"
    elif w == "T1":    sc, why = 20,  "T1"
    else:              sc, why = 10,  f"fallback ({w})"
    n = float(r.get("n_slices") or 0)
    if n < 15: sc -= 15; why += " [ít slice]"
    if float(r.get("obliquity_deg") or 0) > 25: sc -= 10; why += " [cắt xiên]"
    tie = -abs(n - 30) - float(r.get("pixel_spacing_row") or 1.0)*10
    return (sc, tie, why)

rows = []
for (std, pl), g in S.groupby(["StudyInstanceUID","plane"]):
    if pl == "UNKNOWN": continue
    cand = [(score_series(r), r) for _, r in g.iterrows()]
    cand = [(s, r) for s, r in cand if s[0] >= 0]
    if not cand: continue
    (sc, _, why), best = max(cand, key=lambda x: (x[0][0], x[0][1]))
    rows.append({"StudyInstanceUID": std, "plane": pl,
                 "SeriesInstanceUID": best.get("SeriesInstanceUID"),
                 "series_dir": best.get("series_dir"),
                 "SeriesDescription": best.get("SeriesDescription"),
                 "weighting": best.get("weighting"), "fatsat": best.get("fatsat"),
                 "laterality": best.get("laterality"), "n_slices": best.get("n_slices"),
                 "score": sc, "reason": why, "n_candidates": len(cand)})
MANI = pd.DataFrame(rows)

# Mặt phẳng thiếu -> missing-plane mask. Train đủ 3 mặt phẳng không có nghĩa
# test cũng thế: test đến từ site khác.
extra = [{"StudyInstanceUID": s, "plane": p, "SeriesInstanceUID": None,
          "series_dir": None, "score": -1, "reason": "THIẾU mặt phẳng", "n_candidates": 0}
         for s in MANI["StudyInstanceUID"].unique()
         for p in PLANES if p not in set(MANI[MANI["StudyInstanceUID"]==s]["plane"])]
if extra: MANI = pd.concat([MANI, pd.DataFrame(extra)], ignore_index=True)
MANI = MANI.sort_values(["StudyInstanceUID","plane"]).reset_index(drop=True)
MANI.to_csv(OUT/"series_manifest.csv", index=False)

print(f"Manifest: {len(MANI):,} dòng -> series_manifest.csv\n")
print("Lý do chọn:"); print(MANI["reason"].value_counts().to_string())
print("\nSố ứng viên mỗi khe (study, mặt phẳng):")
print(MANI["n_candidates"].value_counts().sort_index().to_string())
print(f"\n{int((MANI['n_candidates']>1).sum()):,} khe có nhiều hơn 1 ứng viên")
print("-> tiebreak thực sự có tác dụng, không phải code thừa.")

## 10 — QC bằng mắt

**Bước này không có script nào thay được.** Nó chặn loại lỗi âm thầm nhất:
train hai tuần trên nhầm chuỗi mà không hay.

In [ ]:
def sort_slices(paths):
    """Sắp theo phép chiếu ImagePositionPatient lên vector pháp tuyến.
    Dùng thứ tự tên file là SAI. InstanceNumber chỉ là fallback."""
    keyed, fb = [], []
    for p in paths:
        try:
            ds = pydicom.dcmread(str(p), stop_before_pixels=True, force=True)
            iop, ipp = getattr(ds,"ImageOrientationPatient",None), getattr(ds,"ImagePositionPatient",None)
            if iop is not None and ipp is not None:
                v = [float(x) for x in iop]
                n = np.cross(np.array(v[:3]), np.array(v[3:]))
                keyed.append((float(np.dot(np.array([float(x) for x in ipp]), n)), p))
            else:
                fb.append((int(getattr(ds,"InstanceNumber",0) or 0), p))
        except Exception:
            fb.append((10**9, p))
    return [p for _,p in sorted(keyed or fb, key=lambda t: t[0])]

def load_slice(path):
    """Xử lý đủ 3 cạm bẫy: rescale, MONOCHROME1, multi-frame."""
    try:
        ds = pydicom.dcmread(str(path), force=True)
        a = ds.pixel_array.astype(np.float32)
        if a.ndim == 3: a = a[len(a)//2]
        a = a*float(getattr(ds,"RescaleSlope",1) or 1) + float(getattr(ds,"RescaleIntercept",0) or 0)
        if str(getattr(ds,"PhotometricInterpretation","")) == "MONOCHROME1":
            a = a.max() - a
        lo, hi = np.percentile(a, [1, 99])
        return np.clip((a-lo)/(hi-lo+1e-8), 0, 1)
    except Exception:
        return None

NCOL = 6
rng = np.random.default_rng(0)
studies = sorted(MANI["StudyInstanceUID"].dropna().unique())
pick = rng.choice(studies, size=min(N_QC_STUDIES, len(studies)), replace=False)

for std in pick:
    rows_ = MANI[MANI["StudyInstanceUID"] == std]
    fig, axes = plt.subplots(3, NCOL, figsize=(NCOL*2.1, 7.6))
    for r, pl in enumerate(PLANES):
        for a in axes[r]: a.axis("off")
        sel = rows_[rows_["plane"] == pl]
        if sel.empty or pd.isna(sel.iloc[0].get("series_dir")):
            axes[r][0].text(.5,.5,f"{pl}\nTHIẾU",ha="center",va="center",color="crimson")
            continue
        row = sel.iloc[0]
        files = sorted(f for f in Path(row["series_dir"]).iterdir() if f.is_file())
        if not files: continue
        ordered = sort_slices(files)
        for c, i in enumerate(np.linspace(0, len(ordered)-1, NCOL).astype(int)):
            img = load_slice(ordered[i])
            if img is None: continue
            axes[r][c].imshow(img, cmap="gray", vmin=0, vmax=1)
            if c == 0:
                axes[r][c].set_title(
                    f"{pl} | {row['weighting']} | fatsat={row['fatsat']} | {row['laterality']}\n"
                    f"{str(row['SeriesDescription'])[:42]}\n{row['reason']}",
                    fontsize=7, loc="left")
    fig.suptitle(str(std)[-46:], fontsize=8)
    plt.tight_layout(); plt.show()

print("""
================== CHECKLIST — NHÌN TỪNG ẢNH Ở TRÊN ==================

  [ ] SAG/COR/AX có đúng là mặt phẳng đó không?
  [ ] Chuỗi fat-sat: dịch có SÁNG và mỡ có TỐI không?
      Nếu tủy xương vẫn sáng trắng -> infer_fatsat SAI, và nhãn
      phù tủy xương của bạn hỏng từ gốc.
  [ ] Có ảnh nào âm bản không? -> bỏ sót MONOCHROME1
  [ ] Nhãn trái/phải có khớp với hình không?
  [ ] Thứ tự slice liên tục hay nhảy cóc?
  [ ] Có localizer/scout nào lọt vào không?

Nếu cả 12 ca đều đạt -> logic chọn chuỗi dùng được.
Nếu không -> sửa score_series() hoặc infer_fatsat() ở cell 3/9 rồi chạy lại.
RẺ HƠN NHIỀU so với phát hiện sau hai tuần train.
""")

## Việc tiếp theo

1. **Xác minh test set có kèm báo cáo không** — quyết định toàn bộ kiến trúc.
2. Nhìn bảng phân bố nhãn ở cell 5. Nhãn nào dưới 5 mẫu dương thì AUC là nhiễu,
   nhưng macro-AUC vẫn tính chúng ngang các nhãn khác.
3. Đọc bằng mắt 10–20 báo cáo ở các ngôn ngữ khác nhau (cell 6) **trước khi**
   viết một dòng code labeler nào.
4. Chốt mm/pixel mục tiêu từ phân bố `PixelSpacing` (cell 7).
5. Chốt cột dùng để GroupKFold theo site (cell 7).
6. Đi qua checklist QC ở cell 10.

`series_meta.parquet`, `study_meta.parquet` và `series_manifest.csv` nằm trong
`/kaggle/working` — kernel tiếp theo mount trực tiếp output này, không cần
upload dataset trung gian.